1. KHAI BÁO THƯ VIỆN

In [14]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

2. PHÂN LOẠI VÀ XỬ LÍ

In [15]:
df = pd.read_csv(r'D:\BUSINESS_PREDICTION\business.csv')

In [16]:
features_to_drop = ['experience_proxy', 'noise_feature_1','noise_feature_2','annual_revenue']
X = df.drop(columns=features_to_drop)
y = df['annual_revenue']

numerical_features = ['age','experience_years','monthly_ad_spend','website_visits','team_size','customer_rating']
categorical_features = ['region','business_type']
ordinal_features = ['subscription_plan']

categories_order = [['Basic','Standard','Premium']]

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
        ('ord', OrdinalEncoder(categories=categories_order), ordinal_features)
    ]
)

3. TIỀN XỬ LÍ DỮ LIỆU

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = list(cat_encoder.get_feature_names_out(categorical_features))
all_features_transformed = numerical_features + encoded_cat_features + ordinal_features

4. HUẤN LUYỆN MODEL

In [19]:
lr_model = LinearRegression()
lr_model.fit(X_train_preprocessed, y_train)
y_pred_lr = lr_model.predict(X_test_preprocessed)

In [20]:
print(f"MAE: {mean_absolute_error(y_test, y_pred_lr):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):,.2f}")
print(f"R² Score: {r2_score(y_test, y_pred_lr):.4f}")

MAE: 15,706.60
RMSE: 19,335.66
R² Score: 0.8510


In [21]:
xgb_model = XGBRegressor(
    n_estimators = 100,
    learning_rate = 0.05,
    max_depth = 5,
    random_state = 42
)
xgb_model.fit(X_train_preprocessed, y_train)
y_pred_xgb = xgb_model.predict(X_test_preprocessed)

In [22]:
print(f"MAE: {mean_absolute_error(y_test, y_pred_xgb):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):,.2f}")
print(f"R² Score: {r2_score(y_test, y_pred_xgb):.4f}")

MAE: 18,380.47
RMSE: 23,061.23
R² Score: 0.7880


In [23]:
importances = xgb_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': all_features_transformed,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)
print(importance_df.to_string(index=False))

                    Feature  Importance
                  team_size    0.269141
           experience_years    0.262617
                        age    0.119392
           monthly_ad_spend    0.067365
          subscription_plan    0.063625
             website_visits    0.037881
            customer_rating    0.036773
               region_Cairo    0.036116
       business_type_Retail    0.031463
         region_Upper_Egypt    0.028131
     business_type_Services    0.021821
business_type_Manufacturing    0.015858
               region_Delta    0.009816


In [24]:
joblib.dump(xgb_model, r'D:\BUSINESS_PREDICTION\xgb_revenue_model.pkl')
joblib.dump(lr_model, r'D:\BUSINESS_PREDICTION\lr_revenue_model.pkl')
joblib.dump(preprocessor, r'D:\BUSINESS_PREDICTION\preprocessor.pkl')

['D:\\BUSINESS_PREDICTION\\preprocessor.pkl']